# OEC — Hugging Face ciphertext read-back + local private-key test

Downloads a chunked public object, verifies the manifest, parts and assembled ciphertext hash, runs your existing local decryptor, then checks the independently known original plaintext hash. Uploads nothing. The notebook never reads key bytes or prints decryptor output. Temporary plaintext/ciphertext are removed after the check.

Requires tools/verify_hf_encrypted_asset.py and huggingface_hub in the selected Python. The manifest protocol must be oec-asset-chunks-v1. Use only an object expected to contain ciphertext.

In [ ]:
from pathlib import Path
import json, subprocess, sys

ROOT = Path.cwd().resolve()
HELPER = ROOT / "tools" / "verify_hf_encrypted_asset.py"
if not HELPER.is_file():
    raise FileNotFoundError(f"Missing {HELPER}; sync the verified OEC source first.")

REPO_ID = ""
MANIFEST_PATH = ""
EXPECTED_CIPHERTEXT_SHA256 = ""
EXPECTED_ORIGINAL_PLAINTEXT_SHA256 = ""
PRIVATE_KEY_PATH = str(Path.home() / ".config" / "oec" / "private_key.pem")
# Replace this template with your real decryptor argv. No shell is used.
DECRYPTOR_ARGV = [
    "python", "/ABSOLUTE/PATH/TO/YOUR/DECRYPTOR.py",
    "--input", "{input}", "--output", "{output}", "--private-key", "{keyfile}",
]

required_values = (
    REPO_ID, MANIFEST_PATH, EXPECTED_CIPHERTEXT_SHA256,
    EXPECTED_ORIGINAL_PLAINTEXT_SHA256,
)
if any(not value.strip() for value in required_values):
    OEC_NOTEBOOK_STATUS = "BLOCKED"
    print("ENCRYPTED_HF_READBACK=BLOCKED_CONFIG_REQUIRED; fill repo, manifest and independent hashes before real verification.")
    print("No remote request, private-key read, or decryption was attempted.")
elif not Path(PRIVATE_KEY_PATH).is_file():
    OEC_NOTEBOOK_STATUS = "BLOCKED"
    print("ENCRYPTED_HF_READBACK=BLOCKED_PRIVATE_KEY_MISSING; private-key contents were not read.")
elif len(DECRYPTOR_ARGV) < 2 or not Path(DECRYPTOR_ARGV[1]).is_file():
    OEC_NOTEBOOK_STATUS = "BLOCKED"
    print("ENCRYPTED_HF_READBACK=BLOCKED_DECRYPTOR_MISSING; configure the existing local decryptor before verification.")
else:
    argv = [
        sys.executable, str(HELPER), "--repo-id", REPO_ID,
        "--manifest-path", MANIFEST_PATH,
        "--expected-ciphertext-sha256", EXPECTED_CIPHERTEXT_SHA256,
        "--expected-plaintext-sha256", EXPECTED_ORIGINAL_PLAINTEXT_SHA256,
        "--private-key-path", PRIVATE_KEY_PATH,
        "--decrypt-command-json", json.dumps(DECRYPTOR_ARGV),
    ]
    print("Running ciphertext integrity and local decrypt round-trip test; key contents are never displayed.")
    result = subprocess.run(argv, cwd=ROOT, check=False)
    print("ENCRYPTED_HF_READBACK_EXIT_CODE=", result.returncode)
    if result.returncode:
        raise RuntimeError("Ciphertext read-back / local decryption / plaintext hash check failed. Do not paste key material.")
    OEC_NOTEBOOK_STATUS = "PASS"
    print("ENCRYPTED_HF_READBACK=PASS")


## Interpret results

PASS means the remote manifest is complete; each part matches its declared size and SHA-256; the assembled object matches both manifest and locally expected ciphertext hashes; your decryptor succeeded; and the decrypted output matches the independently saved original hash. It does not prove the encryption algorithm is secure or inspect other public files and metadata. No upload occurs.